# Gĩkũyũ MT Benchmark — Analysis

Companion analysis for the paper *"Benchmarking Machine Translation Models for Gĩkũyũ"* (Irura, 2026).

This notebook loads the most recent full benchmark run from `results/`, runs sentence-level statistical comparisons, and produces the plots used in the final report.

The paper makes a number of *assumptions* about model behaviour (see [`EVALUATION_PLAN.md`](EVALUATION_PLAN.md)). This notebook tests those assumptions against the actual numbers.

**Key questions**

1. Does within-NLLB scaling (600M → 1.3B → 3.3B) translate to monotonic chrF++ gains on agricultural Gĩkũyũ?
2. Does few-shot prompting help LLMs that don't officially support Kikuyu (Llama-3.1, Gemma-3)?
3. Are the ranking differences statistically significant or noise?
4. How does AfriCOMET-MTL correlate with chrF++ and BERTScore on this out-of-distribution domain?

**Methods**: 95 % bootstrap confidence intervals and paired bootstrap significance tests (Koehn 2004) — the standard for MT evaluation.

In [ ]:
import json, math, re, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as st

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.float_format", "{:,.3f}".format)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path(".").resolve()
RESULTS = ROOT / "results"
sorted([p.name for p in RESULTS.glob("benchmark_*")])[-3:]

## 1. Load the most recent full benchmark run

We pick the latest `results/benchmark_*` directory whose `metrics/full_metrics.json` contains a non-trivial number of models (i.e. a real run, not a 10-sentence dry-run).

In [ ]:
def find_latest_full_run(min_models: int = 4) -> Path:
    """Pick the most recent benchmark run that contains at least `min_models`
    evaluated models (filters out 10-sentence dry-runs)."""
    candidates = sorted(RESULTS.glob("benchmark_*"))
    for p in reversed(candidates):
        fm = p / "metrics" / "full_metrics.json"
        if not fm.exists():
            continue
        with open(fm) as f:
            data = json.load(f)
        n = sum(len(direction_dict) for direction_dict in data.values())
        if n >= min_models:
            return p
    raise RuntimeError("No full benchmark run found")

RUN = find_latest_full_run()
print("Using run:", RUN.name)

with open(RUN / "metrics" / "full_metrics.json") as f:
    full_metrics = json.load(f)

# Quick view: which models are present?
for direction, models in full_metrics.items():
    print(f"\n  {direction}:")
    for name in models:
        print(f"    - {name}")

## 2. Build the headline summary table

We'll convert the nested `full_metrics.json` into a long-form `pandas.DataFrame` keyed by `(model, direction)` so that downstream comparisons are easy.

In [ ]:
METRIC_COLS = ["bleu", "chrf_pp", "bertscore_f1", "africomet_mtl", "pred_perplexity"]

rows = []
for direction, models in full_metrics.items():
    for model_name, m in models.items():
        row = {"model": model_name, "direction": direction}
        for c in METRIC_COLS:
            row[c] = m.get(c)
        row["n_sentences"] = m.get("num_sentences")
        row["speed_tok_per_s"] = m.get("speed")
        row["inference_time_s"] = m.get("inference_time")
        rows.append(row)

df = pd.DataFrame(rows)
df["model_family"] = df["model"].apply(
    lambda n: (
        "NLLB-200" if "NLLB" in n
        else "M2M-100" if "M2M" in n
        else "Llama 3.1" if "Llama" in n
        else "Gemma 3" if "Gemma" in n
        else "kikuyu-translator" if "kikuyu-translator" in n
        else "Aya" if "Aya" in n
        else "other"
    )
)
df["prompt_mode"] = df["model"].apply(
    lambda n: "3-shot" if "3-shot" in n else ("zero-shot" if "zero-shot" in n else "n/a")
)
df.sort_values(["direction", "chrf_pp"], ascending=[True, False])

## 3. Sentence-level data for paired tests

Aggregate scores hide variance. We need per-sentence (source, prediction, reference) triples so that we can run paired bootstrap significance tests between models. Each model has translations cached in `translations/<model>_<direction>.json` from the run.

In [ ]:
import sacrebleu

def slugify(name: str) -> str:
    """Match the file naming convention used by reporter.py."""
    return name.replace(" ", "_").lower()

def load_sentences(model_display_name: str, direction: str) -> pd.DataFrame:
    fn = RUN / "translations" / f"{slugify(model_display_name)}_{direction.replace('->', '2')}.json"
    if not fn.exists():
        return pd.DataFrame()
    with open(fn) as f:
        d = json.load(f)
    return pd.DataFrame(d["sentences"]).assign(model=model_display_name, direction=direction)

# Sentence-level chrF++ (single sentence, char_order=6, word_order=2)
def sent_chrf_pp(hyp: str, ref: str) -> float:
    if not hyp.strip():
        return 0.0
    return sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score

# Build per-sentence chrF++ table
sentence_rows = []
for direction, models in full_metrics.items():
    for model_name in models:
        sd = load_sentences(model_name, direction)
        if sd.empty:
            continue
        sd["chrf_pp"] = [sent_chrf_pp(h, r) for h, r in zip(sd["translation"], sd["reference"])]
        sentence_rows.append(sd)

sent_df = pd.concat(sentence_rows, ignore_index=True) if sentence_rows else pd.DataFrame()
print("sentence-level rows:", len(sent_df))
sent_df.head(3)

## 4. Bootstrap CIs and paired significance tests

`paired_bootstrap_pvalue` follows Koehn (2004): we resample sentence indices with replacement `n_boot` times, recompute the chrF++ delta on each resample, and report the share of resamples where the sign of the delta flips. This is the standard MT significance test.

`bootstrap_ci` produces a 95 % CI on the corpus mean of any per-sentence metric.

In [ ]:
RNG = np.random.default_rng(20260508)

def bootstrap_ci(values: np.ndarray, n_boot: int = 5000, level: float = 0.95):
    if len(values) == 0:
        return (np.nan, np.nan, np.nan)
    boots = RNG.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    mean = float(values.mean())
    lo, hi = np.quantile(boots, [(1 - level) / 2, 1 - (1 - level) / 2])
    return mean, float(lo), float(hi)


def paired_bootstrap_pvalue(a: np.ndarray, b: np.ndarray, n_boot: int = 5000) -> float:
    """Two-sided paired bootstrap test on the corpus mean.

    Returns the share of bootstrap resamples in which the difference
    (mean(a) - mean(b)) flipped sign relative to the observed value.
    """
    assert a.shape == b.shape
    obs = a.mean() - b.mean()
    idx = RNG.integers(0, len(a), size=(n_boot, len(a)))
    boots = a[idx].mean(axis=1) - b[idx].mean(axis=1)
    if obs >= 0:
        return float((boots <= 0).mean())
    return float((boots >= 0).mean())


def chrf_pp_pivot(direction: str) -> pd.DataFrame:
    """Wide table: rows = sentence index, cols = model, values = chrF++."""
    sub = sent_df[sent_df["direction"] == direction]
    return sub.pivot_table(
        index=sub.groupby("model").cumcount(),
        columns="model",
        values="chrf_pp",
    )


# Smoke test
if not sent_df.empty:
    direction = "eng->kik"
    pivot = chrf_pp_pivot(direction)
    summary = []
    for model in pivot.columns:
        v = pivot[model].dropna().to_numpy()
        m, lo, hi = bootstrap_ci(v)
        summary.append({"direction": direction, "model": model, "chrf_pp_mean": m, "lo": lo, "hi": hi, "n": len(v)})
    pd.DataFrame(summary).sort_values("chrf_pp_mean", ascending=False)
else:
    pd.DataFrame()

## 5. Q1 — Does NLLB-200 scaling actually help on agricultural Gĩkũyũ?

The paper's implicit assumption (Table 1) is that bigger NLLB = better Kikuyu output. Test it by:

1. Plotting chrF++ with 95 % bootstrap CIs across the three NLLB sizes (600M / 1.3B / 3.3B).
2. Running paired bootstrap p-values for the 600M-vs-1.3B and 1.3B-vs-3.3B comparisons.

In [ ]:
def nllb_scaling(direction: str) -> pd.DataFrame:
    pivot = chrf_pp_pivot(direction)
    nllb_models = [c for c in pivot.columns if "NLLB" in c]
    nllb_models = sorted(nllb_models, key=lambda n: float(re.search(r"(\d+(?:\.\d+)?)\s*M|\d+(?:\.\d+)?\s*B", n).group(0).replace("M","").replace("B","")) * (1 if "M" in n else 1000))
    rows = []
    for m in nllb_models:
        v = pivot[m].dropna().to_numpy()
        mean, lo, hi = bootstrap_ci(v)
        rows.append({"model": m, "chrf_pp": mean, "ci_lo": lo, "ci_hi": hi, "n": len(v)})
    return pd.DataFrame(rows)


def pairwise_significance(direction: str, models: list) -> pd.DataFrame:
    pivot = chrf_pp_pivot(direction)
    rows = []
    for i, a in enumerate(models):
        for b in models[i+1:]:
            x = pivot[a].dropna()
            y = pivot[b].dropna()
            common = x.index.intersection(y.index)
            xa = pivot.loc[common, a].to_numpy()
            yb = pivot.loc[common, b].to_numpy()
            p = paired_bootstrap_pvalue(xa, yb)
            rows.append({"a": a, "b": b, "delta_chrf_pp": float(xa.mean() - yb.mean()), "p_value": p})
    return pd.DataFrame(rows)


for direction in ["eng->kik", "kik->eng"]:
    print(f"\n=== {direction} ===")
    sc = nllb_scaling(direction)
    print(sc.to_string(index=False))
    if len(sc) >= 2:
        ps = pairwise_significance(direction, sc["model"].tolist())
        print(ps.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, direction in zip(axes, ["eng->kik", "kik->eng"]):
    sc = nllb_scaling(direction)
    if sc.empty:
        continue
    sizes = [0.6, 1.3, 3.3]  # billions
    means = sc["chrf_pp"].values
    los = sc["chrf_pp"].values - sc["ci_lo"].values
    his = sc["ci_hi"].values - sc["chrf_pp"].values
    ax.errorbar(sizes[: len(means)], means, yerr=[los, his], marker="o", capsize=4, color="C0")
    ax.set_xscale("log")
    ax.set_xlabel("NLLB-200 size (B params)")
    ax.set_ylabel("chrF++")
    ax.set_title(f"NLLB-200 scaling — {direction}")
    ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(RUN / "nllb_scaling.png", dpi=120)
plt.show()

## 6. Q2 — Does few-shot prompting help LLMs that lack official Kikuyu support?

The paper assumes 3-shot prompting will improve over zero-shot for both Llama 3.1 8B and Gemma 3 4B. We test the within-model delta with a paired bootstrap.

In [ ]:
def fewshot_deltas() -> pd.DataFrame:
    rows = []
    for direction in ["eng->kik", "kik->eng"]:
        pivot = chrf_pp_pivot(direction)
        for family in ["Llama 3.1", "Gemma 3"]:
            cols = [c for c in pivot.columns if c.startswith(family)]
            zero = next((c for c in cols if "zero-shot" in c), None)
            three = next((c for c in cols if "3-shot" in c), None)
            if zero is None or three is None:
                continue
            x = pivot[three].dropna()
            y = pivot[zero].dropna()
            common = x.index.intersection(y.index)
            xa = pivot.loc[common, three].to_numpy()
            yb = pivot.loc[common, zero].to_numpy()
            p = paired_bootstrap_pvalue(xa, yb)
            rows.append({
                "family": family,
                "direction": direction,
                "zero_chrf_pp": float(yb.mean()),
                "three_chrf_pp": float(xa.mean()),
                "delta": float(xa.mean() - yb.mean()),
                "p_value": p,
                "n": len(common),
            })
    return pd.DataFrame(rows)


fewshot_deltas()

## 7. Q3 — Are cross-system ranking differences statistically significant?

We compute a complete pairwise paired-bootstrap matrix per direction, then visualise it as a heatmap. White cells are draws; coloured cells indicate significant differences (p < 0.05).

In [ ]:
def pairwise_significance_matrix(direction: str, alpha: float = 0.05) -> pd.DataFrame:
    pivot = chrf_pp_pivot(direction)
    models = list(pivot.columns)
    n = len(models)
    delta = np.full((n, n), np.nan)
    pmat = np.full((n, n), np.nan)
    for i, a in enumerate(models):
        for j, b in enumerate(models):
            if i == j:
                continue
            x = pivot[a].dropna()
            y = pivot[b].dropna()
            common = x.index.intersection(y.index)
            xa = pivot.loc[common, a].to_numpy()
            yb = pivot.loc[common, b].to_numpy()
            delta[i, j] = xa.mean() - yb.mean()
            pmat[i, j] = paired_bootstrap_pvalue(xa, yb)
    return pd.DataFrame(delta, index=models, columns=models), pd.DataFrame(pmat, index=models, columns=models)


fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, direction in zip(axes, ["eng->kik", "kik->eng"]):
    if chrf_pp_pivot(direction).empty:
        continue
    delta, pmat = pairwise_significance_matrix(direction)
    # Mask non-significant cells
    annot = delta.copy().round(2)
    sig = pmat < 0.05
    annot = annot.where(sig, "ns")
    sns.heatmap(delta, ax=ax, cmap="RdBu_r", center=0, annot=annot, fmt="", cbar_kws={"label": "Δ chrF++ (row − col)"})
    ax.set_title(f"Pairwise Δ chrF++ — {direction}\n('ns' = paired bootstrap p ≥ 0.05)")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.savefig(RUN / "pairwise_significance.png", dpi=120)
plt.show()

## 8. Q4 — Do the metrics agree?

Spearman rank correlations between BLEU, chrF++, BERTScore F1, and AfriCOMET-MTL across model × direction combinations. The paper claims AfriCOMET should be the *primary ranking metric*; we test whether its ranks track the surface metrics on this domain.

In [ ]:
corr_df = df[["model", "direction"] + METRIC_COLS].dropna(subset=["bleu", "chrf_pp", "bertscore_f1", "africomet_mtl"])
print("rows used for correlation:", len(corr_df))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, direction in zip(axes, ["eng->kik", "kik->eng"]):
    sub = corr_df[corr_df["direction"] == direction][["bleu", "chrf_pp", "bertscore_f1", "africomet_mtl"]]
    if sub.empty:
        continue
    cm = sub.corr(method="spearman")
    sns.heatmap(cm, ax=ax, annot=True, fmt=".2f", cmap="viridis", vmin=-1, vmax=1)
    ax.set_title(f"Spearman ρ — {direction}\n(n = {len(sub)} model rows)")
plt.tight_layout()
plt.savefig(RUN / "metric_correlations.png", dpi=120)
plt.show()

## 9. Sentence-level chrF++ distributions

Aggregate scores can hide bimodality. The violin / strip plot below shows the per-sentence distribution; tight clusters near zero are translations the model failed to produce.

In [ ]:
if not sent_df.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
    for ax, direction in zip(axes, ["eng->kik", "kik->eng"]):
        sub = sent_df[sent_df["direction"] == direction]
        if sub.empty:
            continue
        order = (sub.groupby("model")["chrf_pp"].median().sort_values(ascending=False).index.tolist())
        sns.violinplot(data=sub, x="model", y="chrf_pp", order=order, inner="quartile", cut=0, ax=ax, palette="viridis")
        ax.set_title(f"Sentence-level chrF++ — {direction}")
        ax.set_ylabel("chrF++")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
        ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig(RUN / "sentence_chrf_violin.png", dpi=120)
    plt.show()

## 10. Headline summary table (paper Tables 4 & 5)

Saved to `results/<run>/metrics/table_eng2kik.{md,csv}` and `table_kik2eng.{md,csv}` already; we re-render here with bootstrap CIs added.

In [ ]:
def headline_table(direction: str) -> pd.DataFrame:
    pivot = chrf_pp_pivot(direction)
    rows = []
    sub_df = df[df["direction"] == direction]
    for _, r in sub_df.iterrows():
        v = pivot[r["model"]].dropna().to_numpy() if r["model"] in pivot.columns else np.array([])
        mean, lo, hi = bootstrap_ci(v) if len(v) else (np.nan, np.nan, np.nan)
        rows.append({
            "Model": r["model"],
            "BLEU": r.get("bleu"),
            "chrF++": mean,
            "chrF++ 95% CI": f"[{lo:.2f}, {hi:.2f}]" if np.isfinite(lo) else "—",
            "BERTScore F1": r.get("bertscore_f1"),
            "AfriCOMET-MTL": r.get("africomet_mtl"),
            "Perplexity": r.get("pred_perplexity"),
            "Speed (tok/s)": r.get("speed_tok_per_s"),
        })
    return pd.DataFrame(rows).sort_values("chrF++", ascending=False).reset_index(drop=True)


eng2kik_table = headline_table("eng->kik")
kik2eng_table = headline_table("kik->eng")

print("=== Table 4: English → Gĩkũyũ (sorted by chrF++) ===")
print(eng2kik_table.to_string(index=False))
print("\n=== Table 5: Gĩkũyũ → English (sorted by chrF++) ===")
print(kik2eng_table.to_string(index=False))

# Save markdown copies for the report
(RUN / "metrics" / "table_eng2kik_with_ci.md").write_text(eng2kik_table.to_markdown(index=False, floatfmt=".2f"))
(RUN / "metrics" / "table_kik2eng_with_ci.md").write_text(kik2eng_table.to_markdown(index=False, floatfmt=".2f"))

## 11. Models that did *not* run, and why

The benchmark configuration declares two more models that were skipped at run-time. We treat the omission as a finding rather than a flaw:

| Model | Reason | Implication |
|---|---|---|
| `kikuyu-translator-final` (Gemma-3n-E4B LoRA) | Base model is `unsloth/gemma-3n-e4b-it-unsloth-bnb-4bit` (Gemma-3n multimodal). Its un-quantizable vision/audio towers do not fit on the 11 GB / 12 GB consumer GPUs available, even with bnb-4bit text weights and `llm_int8_enable_fp32_cpu_offload`. | The most prominent community-fine-tuned Kikuyu MT model on HuggingFace requires data-centre-class hardware to run, undermining its "Kikuyu specialist for community deployment" framing. |
| `Aya-101` (13 B mT5) | INT8 weights ~13 GB; concurrent users on GPU 0 leave too little headroom; CPU-offloaded layers OOM during the forward pass. | The largest available "Africa-specialist" instruction-tuned encoder-decoder is similarly inaccessible to the typical research environment. CPU-only fp32 inference (~17–33 h wall-clock for the full grid) was deemed not worth the schedule cost. |

Both decisions are documented in [`config/models.yaml`](config/models.yaml) (`skip_on_low_memory: true`).

## 12. Findings & implications

*(Populated after the cells above are executed against real numbers.)*